# s15 - Adversarial training sebagai defense: evaluasi R6 (SURVEY)

**Klaim R6** (§4.2): adversarial training menaikkan robustness, dengan trade-off
[F16][F17][F32]. Kami bandingkan DUA model global:
- **clean** = dari run R3 (FedAvg biasa) -> `run_S_R3/global_final.npz`
- **adv-trained** = dari run R6 (client pakai `--adv-ratio`) -> `run_S_R6/global_final.npz`

Keduanya dievaluasi: (a) **clean MCC** (test bersih), (b) **robust MCC** (di bawah
PGD constrained, memakai serangan identik s14). Trade-off = berapa clean-acc
dikorbankan untuk menaikkan robust MCC.

**Prasyarat:** run R3 (clean) & R6 (adv) SELESAI di EC2 (`fed_node_survey.py`
`--adv-ratio 0.5 --adv-eps 0.1`). Jika R6 belum ada -> baris adv `[TBD]`.

**Output:** `survey_out/r6_adv_training.csv`.

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','scikit-learn','boto3'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, io, numpy as np
from sklearn.metrics import matthews_corrcoef, accuracy_score
OUTDIR='survey_out'; os.makedirs(OUTDIR,exist_ok=True)
BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'; BASE='unsw-far/federated_survey'
RUN_CLEAN=os.environ.get('RUN_CLEAN','S_R3'); RUN_ADV=os.environ.get('RUN_ADV','S_R6')
ADV_EPS=float(os.environ.get('ADV_EPS','0.1'))
import boto3; s3=boto3.client('s3',region_name=REGION)
def load_npz_s3(key):
    try: obj=s3.get_object(Bucket=BUCKET,Key=key)
    except Exception as e: print('MISS',key,e); return None
    return np.load(io.BytesIO(obj['Body'].read()),allow_pickle=True)

In [ ]:
P={}
for nm in ['cic_train','cic_test','unsw_train','unsw_test']:
    sub='cic' if nm.startswith('cic') else 'unsw'
    d=load_npz_s3(f'{BASE}/data/{sub}/{nm}.npz'); assert d is not None,'jalankan s01 dulu'
    P[nm]=(np.asarray(d['X'],float),np.asarray(d['y'],int))
Xtr=np.vstack([P['cic_train'][0],P['unsw_train'][0]])
Xte=np.vstack([P['cic_test'][0],P['unsw_test'][0]]); yte=np.concatenate([P['cic_test'][1],P['unsw_test'][1]])
zm=Xtr.mean(0); zs=Xtr.std(0); zs[zs==0]=1; Xz=(Xte-zm)/zs
lo=np.percentile((Xtr-zm)/zs,1,axis=0); hi=np.percentile((Xtr-zm)/zs,99,axis=0)
relu=lambda x:np.maximum(0,x); sig=lambda x:1/(1+np.exp(-np.clip(x,-30,30)))
def fwd(W,X):
    a1=X@W[0]+W[1]; z1=relu(a1); a2=z1@W[2]+W[3]; z2=relu(a2); a3=z2@W[4]+W[5]
    return sig(a3).ravel(),(a1,z1,a2,z2)
def dX(W,X,y):
    o,(a1,z1,a2,z2)=fwd(W,X); g=(o-y.astype(float))[:,None]; d2=(g@W[4].T)*(a2>0); d1=(d2@W[2].T)*(a1>0); return d1@W[0].T
def pred(W,X): o,_=fwd(W,X); return (o>=0.5).astype(int)
def pgd(W,X,y,eps,steps=10):
    a=eps/4; Xa=X.copy()
    for _ in range(steps): Xa=np.clip(np.clip(Xa+a*np.sign(dX(W,Xa,y)),X-eps,X+eps),lo,hi)
    return Xa

## 1. Evaluasi clean MCC + robust MCC (PGD) utk tiap model

In [ ]:
def evaluate(run):
    gf=load_npz_s3(f'{BASE}/run_{run}/global_final.npz')
    if gf is None: return None
    W=[gf[f'w{i}'] for i in range(6)]
    clean_mcc=matthews_corrcoef(yte,pred(W,Xz)); clean_acc=accuracy_score(yte,pred(W,Xz))
    atk=(yte==1); Xz2=Xz.copy(); Xz2[atk]=pgd(W,Xz[atk],yte[atk],ADV_EPS)
    robust_mcc=matthews_corrcoef(yte,pred(W,Xz2))
    return dict(clean_mcc=clean_mcc,clean_acc=clean_acc,robust_mcc=robust_mcc)
rows=[]
for name,run in [('clean (FedAvg)',RUN_CLEAN),('adv-trained',RUN_ADV)]:
    m=evaluate(run)
    if m is None:
        print(f'[TBD] {name}: global_final.npz run {run} belum ada')
        rows.append(dict(model=name,clean_mcc=float('nan'),clean_acc=float('nan'),robust_mcc=float('nan')))
    else:
        rows.append(dict(model=name,**m))
        print(f'{name:16s} clean_MCC={m["clean_mcc"]:.3f} clean_acc={m["clean_acc"]:.3f} robust_MCC(PGD)={m["robust_mcc"]:.3f}')

## 2. Trade-off + simpan R6

In [ ]:
import csv
if all(r['robust_mcc']==r['robust_mcc'] for r in rows):
    c=rows[0]; a=rows[1]
    print(f'Robust gain (adv - clean) = {a["robust_mcc"]-c["robust_mcc"]:+.3f} MCC')
    print(f'Clean cost  (adv - clean) = {a["clean_acc"]-c["clean_acc"]:+.3f} acc')
path=os.path.join(OUTDIR,'r6_adv_training.csv')
with open(path,'w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=['model','clean_mcc','clean_acc','robust_mcc']); w.writeheader()
    for r in rows: w.writerow(r)
print('saved',path)
try: s3.upload_file(path,BUCKET,f'{BASE}/results/r6_adv_training.csv'); print('upload OK')
except Exception as e: print('(upload dilewati:',e,')')
print('=== s15 SELESAI. Isi §4.3 baris R6 (robust MCC + Delta clean-acc). ===')